In [1]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

# Import dari file .py yang sudah kita update
from config import (
    FLOOD_SHP_PATH,
    EVAC_GEOJSON_PATH,
    USER_SHP_PATH,
    ELEVATION_TIF_PATH,
    LEARNING_RATE,
    NUM_EPISODES,
)
from device_config import DEVICE
from data_loader import load_evac_candidates_shp, generate_user_coords_from_shp

# Catatan: pastikan data_loader support GeoJSON & return list of dicts
from data_utils import load_flood_polygons, get_elevation_at_point
from model import PolicyNetwork
from flood_trainer import train_rl_model

In [2]:
# %%
# --- PERBAIKAN IMPORT ---
# Jangan pakai 'data_loader', tapi pakai 'data_utils' yang baru dibuat
from data_utils import load_evac_candidates, get_elevation_at_point, load_flood_polygons
from data_loader import (
    generate_user_coords_from_shp,
)  # User tetap dari loader lama tidak apa-apa

print("--- Memuat Dataset ---")

# A. Data Banjir
flood_gdf = load_flood_polygons(FLOOD_SHP_PATH)

# B. Data Evakuasi (PENTING: Gunakan fungsi load_evac_candidates dari data_utils)
# Fungsi ini mengembalikan [{'coord': (lat,lon), 'name': '...'}, ...]
raw_evac_candidates = load_evac_candidates(EVAC_GEOJSON_PATH)

evac_candidates_prep = []
print("   > Pre-processing elevasi kandidat evakuasi...")

# Loop ini sekarang akan BERHASIL karena raw_evac_candidates sudah berbentuk dictionary
for item in raw_evac_candidates:
    lat, lon = item["coord"]  # Ambil koordinat dari key dictionary

    # Ambil elevasi
    elev = get_elevation_at_point(ELEVATION_TIF_PATH, lat, lon)

    # Masukkan data elevasi ke dictionary item
    item["elev"] = elev
    evac_candidates_prep.append(item)

# C. Data User
print("   > Generate user dari pemukiman...")
user_coords = generate_user_coords_from_shp(USER_SHP_PATH, n=100)

print(
    f"✅ Data Siap: {len(user_coords)} Users, {len(evac_candidates_prep)} Titik Evakuasi."
)

--- Memuat Dataset ---
✅ Berhasil memuat 6 poligon banjir.
✅ Berhasil memuat 222 titik evakuasi dari titik_evakuasi_2_dengan_alamat baru.geojson
   > Pre-processing elevasi kandidat evakuasi...
   > Generate user dari pemukiman...
✅ 100 koordinat pengguna berhasil dibuat dari SHP.
✅ Data Siap: 100 Users, 222 Titik Evakuasi.


In [3]:
print("\n--- Setup Model ---")
policy_net = PolicyNetwork(input_dim=6, output_dim=1).to(DEVICE)
optimizer = optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)

print("Model siap dilatih.")


--- Setup Model ---
Model siap dilatih.


In [ ]:
print("\n--- Mulai Training ---")

# Kita panggil fungsi train_rl_model yang ada di file flood_trainer.py
trained_model, reward_history = train_rl_model(
    model=policy_net,
    optimizer=optimizer,
    num_episodes=NUM_EPISODES,
    user_coords=user_coords,
    evac_candidates=evac_candidates_prep,  # Data yang sudah ada elevasinya
    flood_gdf=flood_gdf,
    tif_path=ELEVATION_TIF_PATH,  # Dibutuhkan untuk elevasi User yang random
)


--- Mulai Training ---

🚀 Memulai Training: 200 Episode
   ℹ️ Data: 100 Users, 222 Titik Evakuasi
Episode 5/200 | Avg Reward: -86.97 | Loss: -8.01
   ↳ Contoh: User lari ke 'Gedung Bumi Mandiri'
      (Jarak: 2.39km, Beda Elevasi: -0.3m)
Episode 10/200 | Avg Reward: -87.44 | Loss: -6.78
   ↳ Contoh: User lari ke 'Lembaga Ilmu Pengetshuan Islam & Arab cbg Surabaya (LIPIA SBY)'
      (Jarak: 2.62km, Beda Elevasi: -1.0m)
Episode 15/200 | Avg Reward: -87.45 | Loss: -7.37
   ↳ Contoh: User lari ke 'PT Pupuk Indonesia wilayah 3A Jatim'
      (Jarak: 2.35km, Beda Elevasi: -2.0m)
Episode 20/200 | Avg Reward: -89.10 | Loss: -6.32
   ↳ Contoh: User lari ke 'PT Gudang Garam'
      (Jarak: 2.76km, Beda Elevasi: 5.2m)
Episode 25/200 | Avg Reward: -89.54 | Loss: -5.82
   ↳ Contoh: User lari ke 'Pasar Kembang'
      (Jarak: 1.31km, Beda Elevasi: -2.3m)
Episode 30/200 | Avg Reward: -90.31 | Loss: -6.35
   ↳ Contoh: User lari ke 'Hotel platinum Tunjungan Sby'
      (Jarak: 2.45km, Beda Elevasi: -1.6m)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(reward_history, label="Average Reward")
plt.title("Grafik Performa Agen (Reward per Episode)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
torch.save(trained_model.state_dict(), "model_banjir_final.pt")
print("💾 Model berhasil disimpan.")

In [ ]:
# %%
import matplotlib.pyplot as plt
import contextily as ctx  # Install jika belum: pip install contextily


def visualize_results(model, user_coords, evac_candidates, flood_gdf, tif_path):
    model.eval()

    # Siapkan plot
    fig, ax = plt.subplots(figsize=(12, 10))

    # 1. Plot Banjir
    flood_gdf.plot(ax=ax, color="blue", alpha=0.3, label="Zona Banjir")

    # 2. Plot Semua Kandidat Evakuasi (Titik Kecil Abu-abu)
    evac_lats = [c["coord"][0] for c in evac_candidates]
    evac_lons = [c["coord"][1] for c in evac_candidates]
    ax.scatter(
        evac_lons, evac_lats, c="grey", s=10, alpha=0.5, label="Kandidat Evakuasi"
    )

    # 3. Prediksi Agen untuk 10 User Pertama
    print("Visualisasi 10 User sampel:")
    with torch.no_grad():
        for i, u_loc in enumerate(user_coords[:10]):
            u_lat, u_lon = u_loc

            # --- PREDIKSI ---
            inputs = []
            valid_indices = []
            for idx, cand in enumerate(evac_candidates):
                e_lat, e_lon = cand["coord"]
                # Hitung jarak kasar (Euclidean) biar cepat utk visualisasi, atau pakai OSRM
                dist = (
                    (u_lat - e_lat) ** 2 + (u_lon - e_lon) ** 2
                ) ** 0.5 * 111  # approx km

                inputs.append(
                    [u_lat / 100, u_lon / 100, e_lat / 100, e_lon / 100, dist / 10, 0]
                )
                valid_indices.append(idx)

            tensor_in = torch.tensor(inputs, dtype=torch.float32).to(DEVICE)
            scores = model(tensor_in).squeeze()
            action = torch.argmax(
                scores
            ).item()  # Ambil yang skornya paling tinggi (Greedy)

            real_idx = valid_indices[action]
            chosen = evac_candidates[real_idx]
            c_lat, c_lon = chosen["coord"]

            # Gambar Garis User -> Evakuasi
            ax.plot([u_lon, c_lon], [u_lat, c_lat], "r-", alpha=0.6, linewidth=1)
            ax.scatter(u_lon, u_lat, c="green", s=30, marker="o")  # User = Hijau
            ax.scatter(
                c_lon, c_lat, c="red", s=50, marker="*"
            )  # Tujuan = Bintang Merah

            if i < 3:  # Print info 3 user saja biar ga penuh
                print(
                    f"User {i}: Lari ke {chosen['name']} (Elevasi: {chosen['elev']:.1f}m)"
                )

    # Tambahkan Peta Dasar (Basemap)
    try:
        ctx.add_basemap(
            ax, crs=flood_gdf.crs.to_string(), source=ctx.providers.CartoDB.Positron
        )
    except:
        print("Koneksi internet diperlukan untuk memuat Basemap")

    plt.title("Rute Evakuasi Hasil Training RL")
    plt.legend()
    plt.show()


# Panggil fungsi visualisasi
visualize_results(
    trained_model, user_coords, evac_candidates_prep, flood_gdf, ELEVATION_TIF_PATH
)